# Why the "grid" layout scores so high — and why it's a `recon` artifac

## Setup
Load the optimization module by path (its filename starts with a digit, so it can't be `import`ed
normally). This also pulls in the layout strategies used to build the training set.

In [1]:
import os, sys, glob, json, math
import numpy as np, torch
torch.set_num_threads(2)

_HERE = os.path.abspath("../")
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
    
V6 = "/n/home05/zdimitrov/tambo/TambOpt/detector_optimization_v6"
if V6 not in sys.path:
    sys.path.insert(0, V6)

import importlib.util
spec = importlib.util.spec_from_file_location("de", os.path.join(V6, "04_optimize_differential_evolution_pop.py"))
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)
deepsets_recon = importlib.util.spec_from_file_location("deepsets_recon", os.path.join(V6, "03_train_recon_deepsets.py"))
import modules_v6.detector_strategies_ne as DS
from modules_v6.reconstruction import build_recon_from_ckpt

print("loaded module; DEVICE =", de.DEVICE, " N_DETECTORS =", de.N_DETECTORS)

loaded module; DEVICE = cpu  N_DETECTORS = 100


## Load the frozen models under test + mountain + primaries
These are exactly the artifacts the optimizers see: the dual-species DeepSets surrogate
(`fnn_electron.pt` + `fnn_muon.pt`), the reconstruction MLP (`recon.pt`), and the mountain geometry.
`U_layout(N, E)` is the composite utility that the layout optimizers maximize.

In [2]:
fnn, _ = de._load_models()          # reuse the dual-species surrogate; ignore the MLP recon

# DeepSets reconstruction net written by 03_train_recon_deepsets.py (= RECON_FOLDER + "_deepsets").
# build_recon_from_ckpt takes the LOADED ckpt dict (not a path), n_det, and device.
OUTPUT_FOLDER = de.RECON_FOLDER + "_deepsets"
recon_ckpt = torch.load(os.path.join(OUTPUT_FOLDER, "recon.pt"), map_location=de.DEVICE)
_recon_ds  = build_recon_from_ckpt(recon_ckpt, de.N_DETECTORS, de.DEVICE)
print(f"[load] recon.pt  model={recon_ckpt['config'].get('model_type')}  "
      f"val={recon_ckpt.get('val_total','?')}  <- {OUTPUT_FOLDER}")

# de.utility_of_xy feeds recon a FLAT (B, n_det*4) tensor (the MLP contract), but
# DeepSetsRecon.forward expects (B, n_det, 4). Wrap it so it accepts either shape,
# leaving every test cell below unchanged.
class _ReconFlatAdapter(torch.nn.Module):
    def __init__(self, m, n_det, feat=4):
        super().__init__(); self.m, self.n_det, self.feat = m, n_det, feat
    def forward(self, x):
        if x.dim() == 2:
            x = x.reshape(x.shape[0], self.n_det, self.feat)
        return self.m(x)
recon = _ReconFlatAdapter(_recon_ds, de.N_DETECTORS, 4)

mtn = de.load_tr_mountain(de.GEOMETRY_PATH_RESOLVED, de.GEOMETRY_GROUP, de.DET_KEY,
    east_entry=de.EAST_ENTRY, layer_east_dx=de.LAYER_EAST_DX, n_planes=de.N_PLANES)
prim = torch.load(os.path.join(de.TRAINING_DATASET_FOLDER, "primary.pt")).float()
Ntot = prim.shape[0]

# One fixed 2000-shower batch. U is batch-size independent (see Test 1), so 2000 is plenty.
pb = prim[torch.randperm(Ntot, generator=torch.Generator().manual_seed(42))[:2000]]

def project(N, E):
    return de.project_to_mountain_ne(mtn, torch.as_tensor(N).float(), torch.as_tensor(E).float())

def U_layout(N, E, batch=None, do_project=True):
    """Composite utility U of a (North, East) layout against a primary batch."""
    if do_project:
        N, E = project(N, E)
    u, r, _ = de.utility_of_xy(torch.as_tensor(N).float(), torch.as_tensor(E).float(),
                               pb if batch is None else batch, fnn, recon)
    return float(u)

def clean(scheme):
    N, E = de.sample_initial_layout_ne(mtn, n_units=de.N_DETECTORS, scheme=scheme)
    return project(N, E)

print("models + mountain loaded; primaries:", Ntot)
print("mountain East bbox [%.0f, %.0f]  Up range [%.0f, %.0f]" % (
    mtn.east_lo, mtn.east_hi, mtn.centroids_NUE[:,1].min(), mtn.centroids_NUE[:,1].max()))

[load] fnn_electron.pt  model=deepsets  epoch=101  val=0.40929470789432526
[load] fnn_muon.pt  model=deepsets  epoch=101  val=0.42806699758257183
[load] recon.pt  val=0.10880686880435263
[load] recon.pt  model=deepsets  val=0.0011073681117434587  <- /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/test_v6_run_03_recentered_deepsets
models + mountain loaded; primaries: 1400000
mountain East bbox [-2019, 1182]  Up range [2442, 3886]


## Test 1 — U is a per-event mean, so it does NOT depend on batch size
`U_E` / `U_angle` average over events, so the objective's *level* is independent of how many
showers you score. This rules out the 50k-vs-512 fixed-batch difference (between
`04_optimize_differential_evolution_pop.py` and the others) as a cause of any U-level difference —
only the **layout** matters.

In [3]:
gN, gE = clean("grid")
for B in (256, 512, 2000, 8000):
    b = prim[torch.randperm(Ntot, generator=torch.Generator().manual_seed(42))[:B]]
    print(f"B={B:6d}  U={U_layout(gN, gE, batch=b, do_project=False):+.2f}")

B=   256  U=+209.55
B=   512  U=+208.29
B=  2000  U=+208.62
B=  8000  U=+209.09


## Test 2 — the clean grid scores ~136; center / perturbed score ~25–34
Among hand-built layouts the grid is dramatically higher than a center
cluster or a 1000 m-perturbed grid. (`r_mean` saturates at 1 because `RECONSTRUCT_THRESHOLD=10`
with 100 detectors, so U is driven by angular/energy reconstruction accuracy, not detector count.)

In [4]:
gN, gE = clean("grid"); cN, cE = clean("center")
gen = torch.Generator().manual_seed(1)
pN, pE = project(gN + torch.randn(100, generator=gen)*1000, gE + torch.randn(100, generator=gen)*1000)
for nm, (N, E) in [("clean grid", (gN, gE)), ("clean center", (cN, cE)),
                   ("perturbed grid s=1000", (pN, pE))]:
    print(f"{nm:22s} U={U_layout(N, E, do_project=False):+.2f}")

clean grid             U=+208.62
clean center           U=+24.71
perturbed grid s=1000  U=+131.21


## Test 3 — peak shape: how fast U falls as the grid is jittered
Add i.i.d. Gaussian noise of std σ (metres) to the grid. U decays smoothly but steeply — a ~200 m
jitter (≈4% of the array's ~4600 m span) already roughly halves it. A genuine physics optimum
would not be this fragile; this sharpness is the first hint of a surrogate artifact.

In [5]:
gN, gE = clean("grid")
for s in (0, 25, 50, 100, 200, 400, 800):
    if s == 0:
        print(f"sigma={s:4d}  U={U_layout(gN, gE, do_project=False):+.2f}"); continue
    us = []
    for seed in range(3):
        g = torch.Generator().manual_seed(seed)
        us.append(U_layout(gN + torch.randn(100, generator=g)*s, gE + torch.randn(100, generator=g)*s))
    print(f"sigma={s:4d}  U={np.mean(us):+.2f} (+/-{np.std(us):.1f})")

sigma=   0  U=+208.62
sigma=  25  U=+203.51 (+/-1.6)
sigma=  50  U=+196.75 (+/-2.7)
sigma= 100  U=+187.90 (+/-4.3)
sigma= 200  U=+179.81 (+/-2.4)
sigma= 400  U=+175.92 (+/-1.3)
sigma= 800  U=+164.80 (+/-1.9)


## Test 4 — U of the 7 ACTUAL training layout strategies
The surrogate/recon were trained on these 7 layout families (grid ± jitter, center clusters, and
**three ring radii** up to 1800 m — i.e. the training set DOES include spread-out layouts). Yet
only the tight grid (`grid_jit20`) is high; the spread rings and center clusters all sit at
~56–75. So "spread coverage" by itself does not explain the grid's score.

In [6]:
for name, fn_name, kw in DS._STRATEGIES:
    fn = DS._STRATEGY_FNS[fn_name]
    us = [U_layout(*fn(mtn, n_det=de.N_DETECTORS, rng=np.random.default_rng(s), **kw), do_project=False)
          for s in range(4)]
    print(f"{name:16s} U={np.mean(us):+7.2f} (+/-{np.std(us):4.1f})   {kw}")
print(f"\nclean grid (no jitter): U={U_layout(*clean('grid'), do_project=False):+.2f}")

grid_jit20       U=+205.41 (+/- 0.7)   {'jitter_sigma': 20.0}
grid_jit200      U=+178.75 (+/- 3.0)   {'jitter_sigma': 200.0}
center_gauss200  U=+171.41 (+/- 2.8)   {'sigma': 200.0}
center_gauss400  U=+188.03 (+/- 3.5)   {'sigma': 400.0}
rings_R300       U=+182.68 (+/- 1.1)   {'outer_radius': 300.0, 'n_rings': 5, 'jitter_sigma': 200.0}
rings_R800       U=+190.77 (+/- 0.4)   {'outer_radius': 800.0, 'n_rings': 6, 'jitter_sigma': 200.0}
rings_R1800      U=+180.11 (+/- 3.2)   {'outer_radius': 1800.0, 'n_rings': 8, 'jitter_sigma': 200.0}

clean grid (no jitter): U=+208.62


## Test 6 — Permute the detector ORDER, keep positions identical
Keep the exact same 100 grid positions; only reorder which detector is fed into which slot of
`recon`. A physically correct (permutation-invariant) reconstructor MUST return the same U —
relabeling detectors can't change the physics. It does.

In [8]:
gN, gE = clean("grid")
print(f"grid, CANONICAL order:  U={U_layout(gN, gE, do_project=False):+.2f}")
for s in range(3):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    print(f"  permuted order seed{s}: U={U_layout(gN[p], gE[p], do_project=False):+.2f}")

grid, CANONICAL order:  U=+208.62
  permuted order seed0: U=+208.62
  permuted order seed1: U=+208.62
  permuted order seed2: U=+208.62


## Test 7 — it's NOT coverage: rigid shift + naive lattice
A rigid shift of the whole grid (regularity and coverage preserved) decays U smoothly; and a naive
10×10 lattice with good coverage (91/100 distinct positions after mountain projection) still scores
~22. So poor coverage / clumping is not why the other layouts are low.

In [9]:
gN, gE = clean("grid")
for d in (100, 200, 500, 1000):
    print(f"rigid shift +{d:4d} m North: U={U_layout(gN + d, gE):+.2f}")

ax = torch.linspace(float(gN.min()), float(gN.max()), 10)
ay = torch.linspace(float(gE.min()), float(gE.max()), 10)
GX, GY = torch.meshgrid(ax, ay, indexing="ij")
LN, LE = GX.reshape(-1), GY.reshape(-1)
LNp, LEp = project(LN, LE)
uniq = torch.unique(torch.stack([LNp, LEp], -1).round(), dim=0).shape[0]
print(f"naive 10x10 lattice: U={U_layout(LNp, LEp, do_project=False):+.2f}   "
      f"unique positions after projection = {uniq}/100")

rigid shift + 100 m North: U=+200.62
rigid shift + 200 m North: U=+192.35
rigid shift + 500 m North: U=+176.70
rigid shift +1000 m North: U=+134.83
naive 10x10 lattice: U=+181.25   unique positions after projection = 91/100


## Test 8 — measure `recon`'s permutation-(non)invariance directly
Permuting the detector rows changes recon's raw output by **0%** of its
magnitude — so the deepsets produces an invariant function.

In [10]:
def feats(N, E):
    N, E = project(N, E)
    xy = torch.stack([N, E], -1).unsqueeze(0).expand(pb.shape[0], -1, -1)
    pr = fnn(pb, xy)
    return torch.stack([xy[..., 0], xy[..., 1], pr[..., 0], pr[..., 1]], -1)   # (B, 100, 4)

def invariance(name, N, E):
    f = feats(torch.as_tensor(N).float(), torch.as_tensor(E).float())
    o1 = recon(f.reshape(f.shape[0], -1))
    ds = []
    for s in range(5):
        p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
        o2 = recon(f[:, p, :].reshape(f.shape[0], -1))
        ds.append((o1 - o2).abs().mean().item())
    print(f"{name:12s} mean|delta out|={np.mean(ds):.4f}  (|out|~{o1.abs().mean().item():.3f})  "
          f"rel={np.mean(ds)/o1.abs().mean().item():.3f}")

invariance("grid",       *de.sample_initial_layout_ne(mtn, 100, "grid"))
invariance("rings_R800", *DS.layout_rings(mtn, n_det=100, rng=np.random.default_rng(0),
                                          outer_radius=800.0, n_rings=6, jitter_sigma=200.0))
invariance("center",     *DS.layout_center_gaussian(mtn, n_det=100, rng=np.random.default_rng(0),
                                                     sigma=200.0))

grid         mean|delta out|=0.0000  (|out|~0.488)  rel=0.000
rings_R800   mean|delta out|=0.0000  (|out|~0.487)  rel=0.000
center       mean|delta out|=0.0000  (|out|~0.487)  rel=0.000


## Test 9 — the fair (order-averaged) value of the grid
Physics is order-invariant, so the honest score of a layout is its average over detector orderings.

In [11]:
gN, gE = clean("grid")
canonical = U_layout(gN, gE, do_project=False)
print(f"grid, CANONICAL order:  U={canonical:+.2f}")
us = []
for s in range(15):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    us.append(U_layout(gN[p], gE[p], do_project=False))
us = np.array(us)
print(f"grid, 15 RANDOM orders: mean={us.mean():+.2f}  std={us.std():.2f}  "
      f"min={us.min():+.2f}  max={us.max():+.2f}")
print(f"-> canonical is {(canonical - us.mean())/us.std():.0f} sigma above the order-average")

grid, CANONICAL order:  U=+208.62
grid, 15 RANDOM orders: mean=+208.62  std=0.00  min=+208.62  max=+208.62
-> canonical is 0 sigma above the order-average


## Summary
- Permutation invariariance works 
- the NN is good at reconstruction of all permutaiotons
- shifting the detectors reduces the utility score, which means there is still an issue with the training data